In [1]:
import pandas as pd
import sqlite3

In [2]:
conn=sqlite3.connect('../instacart.db')

craeting table like a lookup table inside SQLite database

In [8]:
tables={
    'orders':'../data/orders.csv',
    'products':'../data/products.csv',
    'aisles':'../data/aisles.csv',
    'departments':'../data/departments.csv',
    'order_products_prior':'../data/order_products__prior.csv',
    'order_products_train':'../data/order_products__train.csv'
}

Loading CSv into the database

In [12]:
for table_name, file_path in tables.items():
    print(f"Loading {table_name}...")
    
    # Read and write in chunks to avoid memory errors
    chunk_size = 500000  # rows per chunk
    first_chunk = True
    total_rows = 0
    
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        if_exists_mode = 'replace' if first_chunk else 'append'
        chunk.to_sql(table_name, conn, if_exists=if_exists_mode, index=False)
        first_chunk = False
        total_rows += len(chunk)
    
    print(f"  -> {total_rows} rows loaded")

print("All tables loaded successfully!")

Loading orders...
  -> 3421083 rows loaded
Loading products...
  -> 49688 rows loaded
Loading aisles...
  -> 134 rows loaded
Loading departments...
  -> 21 rows loaded
Loading order_products_prior...
  -> 32434489 rows loaded
Loading order_products_train...
  -> 1384617 rows loaded
All tables loaded successfully!


In [13]:
pd.read_sql("select * from orders limit 5",conn)

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [14]:
pd.read_sql("SELECT * FROM products LIMIT 5", conn)

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [15]:
pd.read_sql("SELECT * FROM order_products_prior LIMIT 5", conn)

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


## Query 1: Overall Reorder Rate

**Business question:** Out of every product line-item ever added to a cart in this 
dataset, what percentage are reorders (customer has bought this before) vs. 
first-time purchases?

**Why it matters:** This is our baseline reorder rate for the whole dataset. 
Every later segment we analyze (by category, by customer type, by day of week) 
gets compared against this number to see if it's performing above or below average.

**Table used:** `order_products_prior`
**Key column:** `reordered` (1 = reorder, 0 = first-time purchase)

In [16]:
query="""
select
    reordered,
    count(*) as num_orders,
    round(count(*)*100.0/(select count(*) from order_products_prior),2) as pct
from order_products_prior
group by reordered
"""
pd.read_sql(query,conn)

,reordered,num_orders,pct
0,0,13307953,41.03
1,1,19126536,58.97


## Query 2: Reorder Rate by Department

**Business question:** Which product departments have the highest reorder rate? 
This tells us where customer loyalty/habit is strongest vs. where purchases are 
more one-off or exploratory.

**Why it matters:** Departments with high reorder rates are good candidates for 
subscription/auto-reorder features. Departments with low reorder rates might need 
different marketing (discovery-focused) rather than retention-focused campaigns.

**Tables used:** `order_products_prior`, `products`, `departments`
**Key technique:** JOIN across three tables using shared ID columns

In [19]:
query = """
SELECT 
    d.department,
    COUNT(*) as total_orders,
    SUM(opp.reordered) as reorder_count,
    ROUND(SUM(opp.reordered) * 100.0 / COUNT(*), 2) as reorder_rate_pct
FROM order_products_prior opp
JOIN products p ON opp.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY reorder_rate_pct DESC
"""

pd.read_sql(query, conn)

,department,total_orders,reorder_count,reorder_rate_pct
0,dairy eggs,5414016,3627221,67.00
1,beverages,2690129,1757892,65.35
2,produce,9479291,6160710,64.99
3,bakery,1176787,739188,62.81
4,deli,1051249,638864,60.77
5,pets,97724,58760,60.13
6,babies,423802,245369,57.90
7,bulk,34573,19950,57.70
8,snacks,2887550,1657973,57.42
9,alcohol,153696,87595,56.99


**Finding:** Dairy/eggs, beverages, and produce have the highest reorder rates 
   (65-67%) — habitual, routine purchases. Personal care, pantry, and international 
   foods have the lowest (32-37%) — more exploratory/infrequent purchases. This 
   suggests different retention strategies: subscription features for high-reorder 
   categories, discovery/recommendation features for low-reorder categories.

In [ ]:
## Query 3: Order Timing Patterns (Day of Week & Hour of Day)

**Business question:** When during the week and day do customers place the most 
orders? And does reorder behavior spike at different times than first-time 
purchases?

**Why it matters:** This tells us the best times to send retention nudges, push 
notifications, or "time to reorder" reminders — hitting customers when they're 
already in a shopping mindset is far more effective than random timing.

**Table used:** `orders` (has order_dow and order_hour_of_day columns)
**Note:** order_dow is 0-6, but Instacart doesn't publicly confirm which number 
is Sunday vs Monday — we'll treat 0 as the start of the week and just focus on 
the pattern shape, not the exact labels.

In [20]:
query="""
select
    order_dow,
    order_hour_of_day,
    count(*) as total_orders
from orders
group by order_dow,order_hour_of_day
order by order_dow,order_hour_of_day
"""
df_timing=pd.read_sql(query,conn)
df_timing.head(20)

,order_dow,order_hour_of_day,total_orders
0,0,0,3936
1,0,1,2398
2,0,2,1409
3,0,3,963
4,0,4,813
5,0,5,1168
6,0,6,3329
7,0,7,12410
8,0,8,28108
9,0,9,40798
